# Séries temporelles — ARMA, ARIMA

Ce notebook analyse une série temporelle réelle (températures mensuelles Paris
ou données de consommation électrique selon disponibilité) avec les modèles ARMA/ARIMA.

Le cours était en R mais j'ai refait l'analyse en Python parce que je voulais
mieux comprendre les étapes d'identification, surtout le lien entre les ACF/PACF
et l'ordre du modèle.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Données

J'utilise les données de passagers aériens (AirPassengers) — classique mais
bien pour illustrer tendance + saisonnalité + ARIMA.


In [ ]:
# AirPassengers : nombre mensuel de passagers aériens (1949-1960)
# Données publiques, disponibles dans R et statsmodels
try:
    from statsmodels.datasets import get_rdataset
    air = get_rdataset("AirPassengers", "datasets")
    series = pd.Series(
        air.data['value'].values,
        index=pd.date_range('1949-01', periods=144, freq='ME')
    )
except:
    # Simulation d'une série avec tendance et saisonnalité similaires
    t = np.arange(144)
    trend = 100 + 2.5 * t
    seasonal = 30 * np.sin(2 * np.pi * t / 12)
    noise = np.random.normal(0, 15, 144)
    series = pd.Series(
        trend + seasonal + noise,
        index=pd.date_range('1949-01', periods=144, freq='ME')
    )
    print("(Données simulées car dataset non disponible)")

print(f"Longueur : {len(series)}")
print(f"Période  : {series.index[0].strftime('%Y-%m')} → {series.index[-1].strftime('%Y-%m')}")
print(f"Min : {series.min():.0f} | Max : {series.max():.0f} | Moy : {series.mean():.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(series.index, series.values, color='#2c7bb6', linewidth=1.5)
ax.set_xlabel('Date')
ax.set_ylabel('Passagers (milliers)')
ax.set_title('Nombre mensuel de passagers aériens (1949-1960)', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('serie_brute.png', dpi=130)
plt.show()

# Observations immédiates :
# - Tendance croissante claire
# - Saisonnalité annuelle (pics l'été)
# - Variance qui augmente avec le niveau → log-transformation utile

## 2. Transformation logarithmique et décomposition

La variance augmente avec le niveau → série non-stationnaire.
Le log stabilise la variance (transformation de Box-Cox avec λ=0).


In [ ]:
log_series = np.log(series)

# Décomposition additive
decomp = seasonal_decompose(log_series, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
labels = ['Série log', 'Tendance', 'Saisonnalité', 'Résidus']
data_list = [log_series, decomp.trend, decomp.seasonal, decomp.resid]
colors = ['#2c7bb6', '#d7191c', '#4daf4a', '#984ea3']

for ax, d, lbl, col in zip(axes, data_list, labels, colors):
    ax.plot(d.index, d.values, color=col, linewidth=1.4)
    ax.set_ylabel(lbl, fontsize=10)
    ax.grid(True, alpha=0.25)

axes[0].set_title('Décomposition additive (série log)', fontsize=12)
plt.tight_layout()
plt.savefig('decomposition.png', dpi=130)
plt.show()

## 3. Test de stationnarité (Dickey-Fuller augmenté)

ARMA suppose une série stationnaire. On teste ça formellement avec le test ADF.
H0 : présence d'une racine unitaire (non-stationnaire)


In [ ]:
def adf_test(ts, name=""):
    result = adfuller(ts.dropna(), autolag='AIC')
    print(f"{'='*40}")
    print(f"ADF Test — {name}")
    print(f"  Statistique ADF : {result[0]:.4f}")
    print(f"  p-value         : {result[1]:.4f}")
    print(f"  Lags utilisés   : {result[2]}")
    print(f"  Valeurs critiques : { {k: f'{v:.3f}' for k,v in result[4].items()} }")
    if result[1] < 0.05:
        print("  → Rejette H0 : série STATIONNAIRE")
    else:
        print("  → Ne rejette pas H0 : série NON-STATIONNAIRE")

adf_test(log_series, "log(AirPassengers)")

# Différenciation saisonnière (lag 12) puis différenciation d'ordre 1
diff_s = log_series.diff(12).dropna()
adf_test(diff_s, "Diff saisonnière (d=12)")

diff_1s = diff_s.diff(1).dropna()
adf_test(diff_1s, "Diff saisonnière + d=1")

## 4. ACF et PACF — identification de l'ordre

C'est la partie la plus délicate. Les règles :
- PACF coupe après lag p → AR(p)
- ACF coupe après lag q → MA(q)
- Les deux décroissent → ARMA(p,q)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7))

# Série différenciée (stationnaire)
series_stat = diff_1s

plot_acf(series_stat, lags=30, ax=axes[0, 0], title='ACF — série stationnaire', alpha=0.05)
plot_pacf(series_stat, lags=30, ax=axes[0, 1], title='PACF — série stationnaire',
          alpha=0.05, method='ywm')

# Résidus du modèle qu'on va ajuster (anticipé ici pour comparaison)
# On refera ça proprement après ajustement
axes[1, 0].set_title('ACF résidus (après ajustement)', fontsize=11)
axes[1, 1].set_title('PACF résidus (après ajustement)', fontsize=11)

for ax in axes[0]:
    ax.grid(True, alpha=0.25)
    ax.axhline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig('acf_pacf.png', dpi=130, bbox_inches='tight')
plt.show()

## 5. Ajustement ARIMA et sélection du modèle

Je compare plusieurs ordres via l'AIC. On cherche le plus petit AIC.


In [ ]:
# Recherche simple sur quelques ordres ARIMA(p,d,q)
# Note : pour une vraie série saisonnière il faudrait SARIMA
# mais je reste sur ARIMA simple ici pour illustrer la sélection

results_aic = []
orders_to_try = [(p, 1, q) for p in range(4) for q in range(4)]

for order in orders_to_try:
    try:
        mod = ARIMA(log_series, order=order).fit()
        results_aic.append({'order': order, 'AIC': mod.aic, 'BIC': mod.bic})
    except:
        pass

df_aic = pd.DataFrame(results_aic).sort_values('AIC').head(8)
print("Top 8 modèles par AIC :")
print(df_aic.to_string(index=False))

In [ ]:
# Ajuster le meilleur modèle
best_order = df_aic.iloc[0]['order']
print(f"Meilleur ordre sélectionné : ARIMA{best_order}")

best_model = ARIMA(log_series, order=best_order).fit()
print(best_model.summary().tables[1])

In [ ]:
# Diagnostic des résidus
residuals = best_model.resid

fig, axes = plt.subplots(2, 2, figsize=(13, 7))

axes[0,0].plot(residuals.index, residuals.values, color='#2c7bb6', linewidth=1)
axes[0,0].axhline(0, color='red', linewidth=1)
axes[0,0].set_title('Résidus au cours du temps', fontsize=11)
axes[0,0].grid(True, alpha=0.25)

axes[0,1].hist(residuals.dropna(), bins=30, color='#2c7bb6', alpha=0.8, edgecolor='white')
axes[0,1].set_title('Distribution des résidus', fontsize=11)
axes[0,1].grid(True, alpha=0.25, axis='y')

plot_acf(residuals.dropna(), lags=25, ax=axes[1,0],
         title='ACF des résidus', alpha=0.05)
axes[1,0].grid(True, alpha=0.25)

plot_pacf(residuals.dropna(), lags=25, ax=axes[1,1],
          title='PACF des résidus', alpha=0.05, method='ywm')
axes[1,1].grid(True, alpha=0.25)

plt.suptitle(f'Diagnostic résidus — ARIMA{best_order}', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('diagnostic_residus.png', dpi=130, bbox_inches='tight')
plt.show()

# Si ACF/PACF des résidus sont tous dans les bandes → bruit blanc → bon modèle

## 6. Prévision

In [ ]:
n_forecast = 24
forecast = best_model.forecast(steps=n_forecast)
conf_int = best_model.get_forecast(steps=n_forecast).conf_int()

# Repasser en échelle originale
forecast_orig = np.exp(forecast)
ci_lower = np.exp(conf_int.iloc[:, 0])
ci_upper = np.exp(conf_int.iloc[:, 1])

forecast_idx = pd.date_range(
    start=series.index[-1] + pd.DateOffset(months=1),
    periods=n_forecast, freq='ME'
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(series.index[-36:], series.values[-36:], color='#2c7bb6',
        linewidth=2, label='Données historiques')
ax.plot(forecast_idx, forecast_orig.values, color='#d7191c',
        linewidth=2, linestyle='--', label=f'Prévision ({n_forecast} mois)')
ax.fill_between(forecast_idx, ci_lower.values, ci_upper.values,
                color='#d7191c', alpha=0.15, label='IC 95%')
ax.axvline(series.index[-1], color='gray', linewidth=1, linestyle=':')
ax.set_xlabel('Date')
ax.set_ylabel('Passagers (milliers)')
ax.set_title(f'Prévision ARIMA{best_order} — 24 mois', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('prevision.png', dpi=130)
plt.show()

## Ce que j'en retiens

- La **log-transformation** est souvent nécessaire quand la variance augmente avec le niveau.
- Le test **ADF** est indispensable avant d'ajuster un ARMA — c'est facile à oublier.
- L'identification via **ACF/PACF** demande du jugement. En pratique je compare plusieurs
  ordres via l'AIC plutôt que de me fier uniquement aux graphes.
- Le **diagnostic des résidus** est l'étape finale : si l'ACF des résidus montre
  encore de la structure, le modèle n'est pas bon.
- Sur AirPassengers il faudrait SARIMA (saisonnalité marquée) — ARIMA simple
  ne capture pas bien la saisonnalité annuelle.
